In [1]:
import pandas as pd

# 1. 의약품 데이터 로드
df = pd.read_csv('drugs.csv')

# 2. 분석할 핵심 6가지 속성 컬럼 지정
properties = {
    'MW (분자량)': 'Molecular_Weight',
    'logP (지용성)': 'XLogP',
    'TPSA (극성표면적)': 'Polar_Area',
    'HBD (수소결합 공여체)': 'H-Bond_Donor_Count',
    'HBA (수소결합 수용체)': 'H-Bond_Acceptor_Count',
    'Rotatable Bonds': 'Rotatable_Bond_Count'
}

# 3. 속성별 하위 5% ~ 상위 95% 구간 계산 및 출력
print("-" * 50)
print("drugs.csv 기반 분자 속성 데이터 분포 (5% ~ 95% 구간)")
print("-" * 50)

for name, col in properties.items():
    if col in df.columns:
        # 결측치를 제외한 유효 데이터 추출
        valid_data = df[col].dropna()
        
        # 하위 5%, 상위 95% 경계값 계산
        lower_bound = valid_data.quantile(0.05)
        upper_bound = valid_data.quantile(0.95)
        
        print(f"{name:<20}: {lower_bound:>6.1f} ~ {upper_bound:<6.1f}")
print("-" * 50)

--------------------------------------------------
drugs.csv 기반 분자 속성 데이터 분포 (5% ~ 95% 구간)
--------------------------------------------------
MW (분자량)            :  145.2 ~ 811.8 
logP (지용성)          :   -3.7 ~ 5.9   
TPSA (극성표면적)        :   20.3 ~ 269.2 
HBD (수소결합 공여체)      :    0.0 ~ 8.0   
HBA (수소결합 수용체)      :    2.0 ~ 16.0  
Rotatable Bonds     :    0.0 ~ 16.0  
--------------------------------------------------


In [2]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import QED
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

# 1. 데이터 로드
df = pd.read_csv('drugs.csv')
smiles_list = df['SMILES'].dropna().tolist()

print("-" * 60)
print("drugs.csv 데이터셋에 대한 QED 지표 적합성 검증")
print("-" * 60)

# 2. 전체 의약품의 QED 점수 계산
qed_scores = []
for s in smiles_list:
    mol = Chem.MolFromSmiles(s)
    if mol:
        qed_scores.append(QED.qed(mol))

qed_arr = np.array(qed_scores)

# 3. 주요 통계치 산출
mean_qed = np.mean(qed_arr)
median_qed = np.median(qed_arr)
min_qed = np.min(qed_arr)
max_qed = np.max(qed_arr)

# 4. 기준선별 통과율 계산
pass_05 = (sum(1 for q in qed_scores if q >= 0.5) / len(qed_scores)) * 100
pass_06 = (sum(1 for q in qed_scores if q >= 0.6) / len(qed_scores)) * 100

print(f"전체 의약품 수   : {len(qed_scores)} 개")
print(f"QED 평균 점수    : {mean_qed:.3f}")
print(f"QED 중앙값(중심) : {median_qed:.3f}")
print(f"QED 최저 / 최고  : {min_qed:.3f} / {max_qed:.3f}")
print("-" * 60)
print(f"QED >= 0.5 기준 적용 시 실제 의약품 통과율: {pass_05:.1f}%")
print(f"QED >= 0.6 기준 적용 시 실제 의약품 통과율: {pass_06:.1f}%")
print("-" * 60)

------------------------------------------------------------
drugs.csv 데이터셋에 대한 QED 지표 적합성 검증
------------------------------------------------------------
전체 의약품 수   : 15116 개
QED 평균 점수    : 0.502
QED 중앙값(중심) : 0.517
QED 최저 / 최고  : 0.003 / 0.944
------------------------------------------------------------
QED >= 0.5 기준 적용 시 실제 의약품 통과율: 52.5%
QED >= 0.6 기준 적용 시 실제 의약품 통과율: 39.0%
------------------------------------------------------------


전체 15,116개의 승인 의약품을 대상으로 QED를 계산한 결과, 평균 점수는 0.502였으며 보편적인 합격 기준인 'QED 0.5 이상'을 만족하는 약물은 52.5%에 불과.
위음성 발생 위험이 크므로 QED는 사용하지 않기로 함.